# Accelerated Data Science with RAPIDS #

## 04 - Logistic Regression ##

**Table of Contents**
<br>
This notebook uses GPU-accelerated logistic regression to predict infection risk based on features of our population members. This notebook covers the below sections:
1. [Environment](#Environment)
2. [Load Data](#Load-Data)
3. [Logistic Regression](#Logistic-Regression)
    * [Viewing the Regression](#Viewing-the-Regression)
    * [Estimate Probability of Infection](#Estimate-Probability-of-Infection)
4. [Model Explainability](#Model-Explainability)
    * [Show Infection Prevalence is Related to Age](#Show-Infection-Prevalence-is-Related-to-Age)
    * [Exercise #1 - Show Infection Prevalence is Related to Sex](#Exercise-#1---Show-Infection-Prevalence-is-Related-to-Sex)
5. [Making Predictions with Separate Training and Testing Data](#Making-Predictions-with-Separate-Training-and-Test-Data)
    * [Exercise #2 - Fit Logistic Regression Model Using Training Data](#Exercise-#2---Fit-Logistic-Regression-Model-Using-Training-Data)
    * [Use Test Data to Validate Model](#Use-Test-Data-to-Validate-Model)

## Environment ##

In [1]:
import cudf
import cuml
import cupy as cp

## Load Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:

# This cell must run to import the data for the example
import os

dir_path = '/content/drive/MyDrive/Accel_DS_RAPIDS'
file_path = 'part3/data/clean_uk_pop_full.csv'

print(f"Checking existence of directory: {dir_path}")
if os.path.exists(dir_path) and os.path.isdir(dir_path):
    print(f"The directory '{dir_path}' exists.")
else:
    print(f"Creating '{dir_path}'.")
    # Create data folder if it doesn't exist
    os.makedirs(dir_path, exist_ok=True)
absolute_path_to_file = os.path.join(dir_path, file_path)
print(f"\nChecking existence of file: {absolute_path_to_file}")
if os.path.exists(absolute_path_to_file):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' not found.")
    print(f"Downloading ")
    # Download the file from Google Drive\
    import gdown
    url = "https://drive.google.com/uc?id=1VW59ZGVwgOfvHbttIwFagpZQkWBICTKh"
    gdown.download(url,absolute_path_to_file)

Checking existence of directory: /content/drive/MyDrive/Accel_DS_RAPIDS
The directory '/content/drive/MyDrive/Accel_DS_RAPIDS' exists.

Checking existence of file: /content/drive/MyDrive/Accel_DS_RAPIDS/part3/data/clean_uk_pop_full.csv
The file 'part3/data/clean_uk_pop_full.csv' exists.


In [5]:
gdf = cudf.read_csv(absolute_path_to_file, usecols=['age', 'sex', 'infected'])

In [6]:
gdf.dtypes

,0
age,float64
sex,float64
infected,float64


In [7]:
gdf.shape

(58479894, 3)

In [8]:
gdf.head()

,age,sex,infected
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0


## Logistic Regression ##
Logistic regression can be used to estimate the probability of an outcome as a function of some (assumed independent) inputs. In our case, we would like to estimate infection risk based on population members' age and sex.

Below we train a logistic regresion model. We first create a cuML logistic regression instance `logreg`. The `logreg.fit` method takes 2 arguments: the model's independent variables *X*, and the dependent variable *y*. Fit the `logreg` model using the `gdf` columns `age` and `sex` as *X* and the `infected` column as *y*.

In [9]:
logreg = cuml.LogisticRegression()
logreg.fit(gdf[['age', 'sex']], gdf['infected'])

LogisticRegression()

### Viewing the Regression ###
After fitting the model, we could use `logreg.predict` to estimate whether someone has more than a 50% chance to be infected, but since the virus has low prevalence in the population (around 1-2%, in this data set), individual probabilities of infection are well below 50% and the model should correctly predict that no one is individually likely to have the infection.

However, we also have access to the model coefficients at `logreg.coef_` as well as the intercept at `logreg.intercept_`. Both of these values are cuDF Series.

Below we view these values. Notice that changing sex from 0 to 1 has the same effect via the coefficients as changing the age by ~48 years.

In [10]:
type(logreg.coef_)

cudf.core.dataframe.DataFrame

In [11]:
type(logreg.intercept_)

cudf.core.series.Series

In [12]:
logreg_coef = logreg.coef_
logreg_int = logreg.intercept_

print("Coefficients: [age, sex]")
print([logreg_coef[0], logreg_coef[1]])

print("Intercept:")
print(logreg_int[0])

Coefficients: [age, sex]
[0    0.014861
Name: 0, dtype: float64, 0    0.695666
Name: 1, dtype: float64]
Intercept:
-5.222369426308749


### Estimate Probability of Infection ###
As with all logistic regressions, the coefficients allow us to calculate the logit for each; from that, we can calculate the estimated percentage risk of infection.

**Note**: Remembering that a 1 indicates 'infected', we assign that class' probability to a new column in the original dataframe.

In [13]:
class_probs = logreg.predict_proba(gdf[['age', 'sex']])
class_probs

,0,1
0,0.994634,0.005366
1,0.994634,0.005366
2,0.994634,0.005366
3,0.994634,0.005366
4,0.994634,0.005366
...,...,...
58479889,0.960428,0.039572
58479890,0.960428,0.039572
58479891,0.960428,0.039572
58479892,0.960428,0.039572


In [14]:
gdf['risk'] = class_probs[1]

Looking at the original records with their new estimated risks, we can see how estimated risk varies across individuals.

In [15]:
gdf.take(cp.random.choice(gdf.shape[0], size=5, replace=False))

,age,sex,infected,risk
13608562,36.0,0.0,0.0,0.009127
1616820,4.0,0.0,0.0,0.005692
7004170,19.0,0.0,0.0,0.007104
24809035,67.0,0.0,0.0,0.014390
51227240,60.0,1.0,0.0,0.025704


## Model Explainability ##
Model explainability refers to the ability to understand and explain the decisions and reasoning underlying the predictions from machine learning models. It can be achieved by investigating how the feature variables are related to the target variable.

### Show Infection Prevalence is Related to Age ###
The positive coefficient on age suggests that the virus is more prevalent in older people, even when controlling for sex.

For this exercise, show that infection prevalence has some relationship to age by printing the mean `infected` values for the oldest and youngest members of the population when grouped by age:

In [16]:
# %load solutions/risk_by_age
age_groups = gdf[['age', 'infected']].groupby(['age'])
print(age_groups.mean().head())
print(age_groups.mean().tail())


      infected
age           
76.0  0.021928
74.0  0.021807
79.0  0.022518
86.0  0.023417
16.0  0.010386
      infected
age           
57.0  0.019792
53.0  0.019236
84.0  0.023350
65.0  0.020792
24.0  0.013022


### Exercise #1 - Show Infection Prevalence is Related to Sex ###
Similarly, the positive coefficient on sex suggests that the virus is more prevalent in people with sex = `1` (females), even when controlling for age.

**Instructions**: <br>
* Modify the `<FIXME>` only and execute the below cell to show that infection prevalence has some relationship to sex by printing the mean `infected` values for the population when grouped by sex. .

In [ ]:
sex_groups = gdf[[<<<<FIXME>>>>, <<<<FIXME>>>>]].groupby([<<<<FIXME>>>>])
sex_groups.mean()

,infected
sex,
0.0,0.010140
1.0,0.020713


<details style='border:1px solid #d0d7de; border-radius:10px; padding:0.45em 0.8em; background:linear-gradient(180deg,#f8fbff 0%,#ffffff 100%); box-shadow:0 1px 3px rgba(0,0,0,0.06);'>
<summary style='font-weight:600; color:#0b57d0; cursor:pointer;'>Show Solution</summary>

```python
sex_groups = gdf[['sex', 'infected']].groupby(['sex'])
sex_groups.mean()
```
</details>

## Making Predictions with Separate Training and Test Data ##
The typical process involves training the model on the training set, then using the test set to evaluate its performance. This provides a more realistic assessment of how well the model will perform on new, unseen data in real-world applications. By testing on a separate dataset, you can detect if your model is **overfitting** to the training data. Overfitting occurs when a model performs well on training data but poorly on new data. In many cases, you don't have access to truly new data, so splitting your existing data simulates this scenario.

cuML gives us a simple method for producing paired training/testing data:

In [17]:
X_train, X_test, y_train, y_test  = cuml.train_test_split(gdf[['age', 'sex']], gdf['infected'], train_size=0.9)

### Exercise #2 - Fit Logistic Regression Model Using Training Data ###

**Instructions**: <br>
* Execute the below cell to create a new logistic regression model `logreg`
* Modify the `<FIXME>` only and execute the cell below to fit the new model with the *X* and *y* training data just created.

In [18]:
logreg = cuml.LogisticRegression()

In [20]:
logreg.fit(<<<<FIXME>>>>, <<<<FIXME>>>>)

LogisticRegression()

<details style='border:1px solid #d0d7de; border-radius:10px; padding:0.45em 0.8em; background:linear-gradient(180deg,#f8fbff 0%,#ffffff 100%); box-shadow:0 1px 3px rgba(0,0,0,0.06);'>
<summary style='font-weight:600; color:#0b57d0; cursor:pointer;'>Show Solution</summary>

```python
logreg.fit(X_train, y_train)
```
</details>


### Use Test Data to Validate Model ###
We can now use the same procedure as above to predict infection risk using the test data:

In [21]:
y_test_pred = logreg.predict_proba(X_test, convert_dtype=True)[1]
y_test_pred.index = X_test.index
y_test_pred

58425474    0.039568
13351222    0.009124
11911022    0.008602
25803360    0.015259
46351712    0.021586
              ...   
45121710    0.020663
11762928    0.008476
37287682    0.015206
5891255     0.006793
7587025     0.007313
Name: 1, Length: 5847990, dtype: float64

As we saw before, very few people are actually infected in the population, even among the highest-risk groups. As a simple way to check our model, we split the test set into above-average predicted risk and below-average predicted risk, then observe that the prevalence of infections correlates closely to those predicted risks.

In [22]:
test_results = cudf.DataFrame()
test_results['age'] = X_test['age']
test_results['sex'] = X_test['sex']
test_results['infected'] = y_test
test_results['predicted_risk'] = y_test_pred

test_results['high_risk'] = test_results['predicted_risk'] > test_results['predicted_risk'].mean()

risk_groups = test_results.groupby('high_risk')
risk_groups.mean()

,age,sex,infected,predicted_risk
high_risk,,,,
True,56.359577,0.880541,0.023767,0.023224
False,29.219316,0.254131,0.010015,0.010281


Finally, in a few milliseconds, we can do a two-tier analysis by sex and age:

In [23]:
%%time
s_groups = test_results[['sex', 'age', 'infected', 'predicted_risk']].groupby(['sex', 'age'])
s_groups.mean()

CPU times: user 24.8 ms, sys: 13.5 ms, total: 38.3 ms
Wall time: 58.1 ms


infected  predicted_risk
sex age                           
0.0 69.0  0.014211        0.014819
    13.0  0.005592        0.006499
1.0 74.0  0.028527        0.031453
    61.0  0.027468        0.026069
    55.0  0.026814        0.023897
...            ...             ...
0.0 25.0  0.009045        0.007758
    31.0  0.010698        0.008476
    89.0  0.016436        0.019849
    85.0  0.014746        0.018724
1.0 86.0  0.028450        0.037368

[182 rows x 2 columns]